# Fase II · Inventario del archivo NASA IMS Bearing Dataset

Se registra la información obtenida a partir de la identificación de las condiciones anómalas: experimentos disponibles, rodamientos, canales, frecuencia de muestreo, momentos de adquisición y eventos finales documentados.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path.cwd().parent / "data" / "raw"
INTERIM = RAW.parent / "interim"
INTERIM.mkdir(exist_ok=True)

EXPERIMENTOS = {
    "Experimento 1": RAW / "1st_test" / "1st_test",
    "Experimento 2": RAW / "2nd_test" / "2nd_test",
    "Experimento 3": RAW / "3rd_test" / "4th_test" / "txt",
}

## 1. Estructura de carpetas

Se recorren las carpetas de `data/raw` y se cuenta cuántos archivos tiene cada una. Se hace antes de cualquier análisis porque no se conocía cómo había organizado los datos el autor en Kaggle, y todo el código posterior depende de esa estructura.

In [1]:
from pathlib import Path
import pandas as pd

# El notebook se ejecuta desde la carpeta notebooks/, por eso subimos un nivel
RAW = Path.cwd().parent / "data" / "raw"
print("¿Existe la carpeta de datos?", RAW.exists())

# Recorre todas las subcarpetas y cuenta cuántos archivos tiene cada una
for carpeta in sorted(p for p in RAW.rglob("*") if p.is_dir()):
    archivos = [f for f in carpeta.iterdir() if f.is_file()]
    print(f"{carpeta.relative_to(RAW)}  ->  {len(archivos)} archivos")

¿Existe la carpeta de datos? True
1st_test  ->  0 archivos
1st_test\1st_test  ->  2156 archivos
2nd_test  ->  0 archivos
2nd_test\2nd_test  ->  984 archivos
3rd_test  ->  0 archivos
3rd_test\4th_test  ->  0 archivos
3rd_test\4th_test\txt  ->  6324 archivos


Los datos están completos y organizados en tres experimentos: 2.156 archivos en el primero, 984 en el segundo y 6.324 en el tercero (9.464 en total). La carpeta del tercer experimento se llama `4th_test/txt`, una inconsistencia de nombres heredada de la distribución original que conviene documentar.

## 2. Formato de un archivo de ejemplo

Se lee una sola captura para confirmar su formato real contra lo que dice la documentación, antes de procesar miles de archivos. Los archivos se identifican por su nombre, que es la fecha y hora de la captura.

In [2]:
# Los archivos de IMS se nombran con la fecha de captura, ej: 2003.10.22.12.06.24
archivos = sorted(f for f in RAW.rglob("20*") if f.is_file())
print("Total de archivos de señal encontrados:", len(archivos))

ejemplo = archivos[0]
print("Archivo de ejemplo:", ejemplo.relative_to(RAW))

df = pd.read_csv(ejemplo, sep="\t", header=None)
print("Dimensiones (filas, columnas):", df.shape)
df.head()

Total de archivos de señal encontrados: 9464
Archivo de ejemplo: 1st_test\1st_test\2003.10.22.12.06.24
Dimensiones (filas, columnas): (20480, 8)


,0,1,2,3,4,5,6,7
0,-0.022,-0.039,-0.183,-0.054,-0.105,-0.134,-0.129,-0.142
1,-0.105,-0.017,-0.164,-0.183,-0.049,0.029,-0.115,-0.122
2,-0.183,-0.098,-0.195,-0.125,-0.005,-0.007,-0.171,-0.071
3,-0.178,-0.161,-0.159,-0.178,-0.100,-0.115,-0.112,-0.078
4,-0.208,-0.129,-0.261,-0.098,-0.151,-0.205,-0.063,-0.066


Cada archivo es texto separado por tabulaciones, sin encabezados. La primera captura del experimento 1 tiene 20.480 filas (un segundo de vibración a 20 kHz) y 8 columnas (dos acelerómetros por cada uno de los cuatro rodamientos), tal como indica la documentación.

## 3. Tabla de inventario por experimento

Se resume cada experimento (capturas, canales, inicio, fin, duración e intervalos) usando solo los nombres de los archivos, lo que es rápido porque no requiere leer los 6,5 GB. Es la descripción básica que cualquier lector del proyecto necesita.

In [3]:
# Ruta de cada experimento (el tercero viene en la carpeta 4th_test\txt)
EXPERIMENTOS = {
    "Experimento 1": RAW / "1st_test" / "1st_test",
    "Experimento 2": RAW / "2nd_test" / "2nd_test",
    "Experimento 3": RAW / "3rd_test" / "4th_test" / "txt",
}

filas = []
intervalos_por_exp = {}

for nombre, carpeta in EXPERIMENTOS.items():
    archivos_exp = sorted(f for f in carpeta.iterdir() if f.is_file())

    # Convertimos el nombre de cada archivo en fecha y hora
    fechas = pd.to_datetime([f.name for f in archivos_exp], format="%Y.%m.%d.%H.%M.%S")

    # Minutos transcurridos entre capturas consecutivas
    intervalos = pd.Series(fechas).diff().dropna().dt.total_seconds() / 60
    intervalos_por_exp[nombre] = intervalos

    # Leemos solo el primer archivo para saber cuántos canales tiene
    n_canales = pd.read_csv(archivos_exp[0], sep="\t", header=None).shape[1]

    filas.append({
        "experimento": nombre,
        "capturas": len(archivos_exp),
        "canales": n_canales,
        "inicio": fechas.min(),
        "fin": fechas.max(),
        "duración (días)": round((fechas.max() - fechas.min()).total_seconds() / 86400, 1),
        "intervalo típico (min)": intervalos.median(),
        "intervalo máximo (min)": round(intervalos.max(), 1),
    })

inventario = pd.DataFrame(filas).set_index("experimento")
inventario

,capturas,canales,inicio,fin,duración (días),intervalo típico (min),intervalo máximo (min)
experimento,,,,,,,
Experimento 1,2156,8,2003-10-22 12:06:24,2003-11-25 23:39:56,34.5,10.0,8905.6
Experimento 2,984,4,2004-02-12 10:32:39,2004-02-19 06:22:39,6.8,10.0,10.0
Experimento 3,6324,4,2004-03-04 09:27:46,2004-04-18 02:42:55,44.7,10.0,452.0


El experimento 1 dura 34,5 días con 8 canales; el 2 dura 6,8 días y el 3 dura 44,7 días, ambos con 4 canales. El intervalo típico entre capturas es de 10 minutos en los tres, pero el intervalo máximo del experimento 1 (unos 8.906 minutos, más de seis días) alerta sobre interrupciones largas.

## 4. Intervalos entre capturas

Se cuenta cuántas veces aparece cada intervalo entre capturas consecutivas. Como el objetivo es estimar el tiempo hasta la falla, la regularidad del eje temporal es crítica, y esta tabla revela cambios de protocolo y huecos.

In [4]:
for nombre, intervalos in intervalos_por_exp.items():
    print(f"\n{nombre}: frecuencia de los intervalos entre capturas (minutos)")
    print(intervalos.round().value_counts().sort_index().to_string())


Experimento 1: frecuencia de los intervalos entre capturas (minutos)
0.0          1
2.0          2
3.0          2
4.0          4
5.0        148
6.0          4
7.0          2
8.0          1
9.0          5
10.0      1962
40.0         1
41.0         1
60.0         1
72.0         1
100.0        1
102.0        1
116.0        1
123.0        1
135.0        1
144.0        1
178.0        1
181.0        1
226.0        1
426.0        1
650.0        1
701.0        1
814.0        1
830.0        1
834.0        1
1230.0       1
2042.0       1
5436.0       1
5790.0       1
8906.0       1

Experimento 2: frecuencia de los intervalos entre capturas (minutos)
10.0    983

Experimento 3: frecuencia de los intervalos entre capturas (minutos)
1.0         1
2.0         1
5.0         2
10.0     6315
70.0        1
270.0       1
440.0       1
452.0       1


El experimento 2 es perfectamente regular: todos sus intervalos son de 10 minutos. El 3 es casi regular (6.315 de 6.323 intervalos de 10 minutos). El 1 es irregular: tiene 148 capturas cada 5 minutos (cambio de protocolo), varios intervalos cortos e irregulares y muchas interrupciones largas.

## 5. Pausas en el registro

Se lista cada pausa mayor a 30 minutos (tres intervalos normales) con su inicio, fin y duración, y se calcula el tiempo con registro continuo. Esto cuantifica cuánto del tiempo de cada prueba realmente tiene datos.

In [5]:
UMBRAL = 30  # minutos: consideramos "pausa" cualquier intervalo mayor a esto

for nombre, carpeta in EXPERIMENTOS.items():
    nombres = sorted(f.name for f in carpeta.iterdir() if f.is_file())
    fechas = pd.Series(pd.to_datetime(nombres, format="%Y.%m.%d.%H.%M.%S"))
    minutos = fechas.diff().dt.total_seconds() / 60

    pausas = pd.DataFrame({
        "desde": fechas.shift(1),
        "hasta": fechas,
        "horas": (minutos / 60).round(1),
    })[minutos > UMBRAL]

    dias_calendario = (fechas.max() - fechas.min()).total_seconds() / 86400
    dias_efectivos = minutos[minutos <= UMBRAL].sum() / 1440

    print(f"\n{nombre}: {len(pausas)} pausas | "
          f"{dias_calendario:.1f} días calendario | {dias_efectivos:.1f} días con registro continuo")
    if len(pausas) > 0:
        display(pausas.reset_index(drop=True))


Experimento 1: 24 pausas | 34.5 días calendario | 14.2 días con registro continuo


,desde,hasta,horas
0,2003-10-23 10:14:13,2003-10-29 14:39:46,148.4
1,2003-10-30 03:39:46,2003-10-31 00:09:46,20.5
2,2003-10-31 10:09:46,2003-10-31 12:05:51,1.9
3,2003-11-01 23:51:44,2003-11-03 09:53:55,34.0
4,2003-11-03 14:21:44,2003-11-07 14:51:44,96.5
5,2003-11-07 22:41:44,2003-11-08 10:22:19,11.7
6,2003-11-09 18:15:58,2003-11-09 22:02:15,3.8
7,2003-11-10 16:25:58,2003-11-14 11:02:17,90.6
8,2003-11-14 17:12:17,2003-11-14 17:52:14,0.7
9,2003-11-16 20:28:46,2003-11-17 10:18:56,13.8



Experimento 2: 0 pausas | 6.8 días calendario | 6.8 días con registro continuo

Experimento 3: 4 pausas | 44.7 días calendario | 43.9 días con registro continuo


,desde,hasta,horas
0,2004-03-15 07:32:46,2004-03-15 12:02:57,4.5
1,2004-03-19 03:05:20,2004-03-19 10:37:17,7.5
2,2004-04-04 01:51:57,2004-04-04 03:01:57,1.2
3,2004-04-15 03:01:57,2004-04-15 10:21:53,7.3


El experimento 1 tiene 24 pausas y solo 14,2 de sus 34,5 días tienen registro continuo (41 %); la mayor pausa dura 148 horas. El experimento 2 no tiene pausas. El 3 tiene 4 pausas, la mayor de 7,5 horas, y conserva 43,9 de sus 44,7 días. Las gráficas deben usar fechas reales y cortarse en las pausas.

## 6. Intervalos cortos tras reanudar el registro

Se revisan los pares de capturas separados por menos de 4 minutos, porque podían ser duplicados o errores, y se comparan con los finales de las pausas.

In [6]:
carpeta = EXPERIMENTOS["Experimento 1"]
nombres = sorted(f.name for f in carpeta.iterdir() if f.is_file())
fechas = pd.Series(pd.to_datetime(nombres, format="%Y.%m.%d.%H.%M.%S"))
segundos = fechas.diff().dt.total_seconds()

cortos = pd.DataFrame({"archivo_anterior": pd.Series(nombres).shift(1),
                       "archivo": nombres,
                       "segundos": segundos})[segundos < 240]
cortos

,archivo_anterior,archivo,segundos
1,2003.10.22.12.06.24,2003.10.22.12.09.13,169.0
782,2003.11.09.22.02.15,2003.11.09.22.05.58,223.0
1237,2003.11.17.10.18.56,2003.11.17.10.22.30,214.0
1439,2003.11.19.11.04.17,2003.11.19.11.06.07,110.0
1508,2003.11.20.11.50.35,2003.11.20.11.54.03,208.0
1615,2003.11.20.23.40.19,2003.11.20.23.44.03,224.0
1857,2003.11.22.23.14.58,2003.11.22.23.16.56,118.0
1969,2003.11.23.20.01.11,2003.11.23.20.01.24,13.0
2052,2003.11.24.12.53.55,2003.11.24.12.57.32,217.0


Casi todos los intervalos cortos ocurren justo después de que termina una pausa, lo que indica que al reiniciar la adquisición se tomaba una captura inicial antes de retomar el ciclo normal: son parte del procedimiento, no errores. El único caso sospechoso es el de 13 segundos, que se revisa en el notebook de integridad.